입력 토큰

  ↓

Embedding

  ↓

Positional Encoding

  ↓

[Encoder Block] 여러 번 반복

  ├─ Multi-Head Self-Attention

  ├─ Add & Norm

  ├─ Feed Forward

  └─ Add & Norm

  ↓

출력


- Transformer를 공부할 때, 제일 중요한 건 수식보다는 shape
    - 보통 (batch_size, seq_len, d_model)를 둠.

## Embedding

In [1]:
import torch
import torch.nn as nn

# 예시 설정
vocab_size = 20
d_model = 8

embedding = nn.Embedding(vocab_size, d_model)
## 20개의 큰 각각에 대해 8개의 숫자를 가진 벡터를 하나씩 저장하는 표
## d_model은 임베딩 벡터의 길이, 각 토큰을 몇 차원 벡터로 표현할 것인지

# batch_size=2, seq_len=5
x = torch.tensor([
    [1, 3, 5, 7, 9],
    [2, 4, 6, 8, 0]
])

out = embedding(x)

print("입력 shape:", x.shape)       # (2, 5)
print("임베딩 후 shape:", out.shape) # (2, 5, 8)
print(out)

입력 shape: torch.Size([2, 5])
임베딩 후 shape: torch.Size([2, 5, 8])
tensor([[[ 1.7021, -0.9006, -0.2275, -0.4576, -1.1971,  0.0715, -1.0377,
           0.3799],
         [ 0.0417,  1.1633,  0.2167, -0.0382,  0.5217, -1.1143, -0.6673,
           1.5639],
         [-2.2401,  2.0462, -0.1140, -0.8041, -1.8107, -0.0796, -0.2293,
          -0.8387],
         [ 0.5440, -0.8331,  0.1016,  0.3465, -1.0284, -0.1000,  0.4520,
          -0.0439],
         [ 0.0402, -0.3474,  0.1839,  0.3405, -0.2794, -0.4390,  0.6481,
           0.2549]],

        [[-1.2067,  0.6772,  0.2781, -0.2739,  0.7002, -0.1172,  0.0179,
          -0.5608],
         [ 1.1253, -0.1652,  0.5000,  0.3211,  0.4565,  0.0751, -0.9837,
           0.2820],
         [-0.7280, -0.5764,  0.1391, -0.0373,  0.3549,  1.3625,  0.7336,
          -1.3955],
         [-0.2334, -0.5831, -0.6350,  1.2295, -0.3949,  1.7749,  0.3380,
           0.0683],
         [ 0.4102, -0.1497, -0.1585, -1.9822, -0.7787,  0.4142, -0.1142,
           0.7933]]], gr

원래 입력은 단어 번호만 있었기 때문에 (2,5) -> Embedding을 통과하면 각 번호가 벡터가 되므로 (2,5,8)이 됨.

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("Using device:", device)
embedding = embedding.to(device)

Using device: cpu


## Positional Encoding

- Transformer는 RNN과 달리 순서를 자동으로 기억하지 못함. 그래서 위치 정보를 따로 더해줘야 함.

1. Learned Positional Encoding
 - 위치 벡터를 학습함.
 - 학습한 최대 길이 밖으로 일반화가 약할 수 있음.

2. Sinusoidal Positional Encoding
- 수식으로 직접 생성(sine, cosine)
- 더 큰 시퀀스로 extrapolation이 상대적으로 쉬움.
- 위치 간 상대적 패턴이 수학적으로 드러남

1. Learned Positional Encoding

In [3]:
class PositionalEmbedding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()
        self.pos_embedding = nn.Embedding(max_len, d_model)

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        batch_size, seq_len, d_model = x.shape

        positions = torch.arange(seq_len, device=x.device).unsqueeze(0)
        # positions shape: (1, seq_len)

        pos_embed = self.pos_embedding(positions)
        # pos_embed shape: (1, seq_len, d_model)

        return x + pos_embed

In [4]:
pos_layer = PositionalEmbedding(max_len=10, d_model=8)

x_embed = embedding(x)
x_pos = pos_layer(x_embed)

print("위치정보 추가 후 shape:", x_pos.shape)

위치정보 추가 후 shape: torch.Size([2, 5, 8])


2. Sinusoidal postional encoding

In [ ]:
import torch
import torch.nn as nn
import math

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, max_len, d_model):
        super().__init__()

        pe = torch.zeros(max_len, d_model)  # (max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)  # (max_len, 1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)  # 짝수 인덱스
        pe[:, 1::2] = torch.cos(position * div_term)  # 홀수 인덱스

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        # unsqueeze한 이유? x의 shape을 맞춰주려고(batch_size)
        self.register_buffer("pe", pe)
        # register_buffer란?
        ## 학습되지는 않지만, 모델의 일부로 관리되어야 하는 텐서(device)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len, :]

## Self-Attention

$Attention(Q, K, V) = softmax({\dfrac {QK^T}{\sqrt {d_k}}})~V$

### Single Head Self-Attention

In [10]:
import torch
import torch.nn

class Mymodel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(4,4)

        pe = torch.ones(1,10,4)
        self.register_buffer("pe",pe)

model = Mymodel()

print("parameters:")
for name,p in model.named_parameters():
    print(name)

print("buffers:")
for name,b in model.named_buffers():
    print(name)

parameters:
linear.weight
linear.bias
buffers:
pe


In [6]:
import math

class SelfAttention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)

        Q = self.W_q(x)  # (batch_size, seq_len, d_model)
        K = self.W_k(x)  # (batch_size, seq_len, d_model)
        V = self.W_v(x)  # (batch_size, seq_len, d_model)

        print("Q shape:", Q.shape)
        print("K shape:", K.shape)
        print("V shape:", V.shape)

        # K.transpose(-2, -1): (batch_size, d_model, seq_len)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(x.size(-1))
        # scores: (batch_size, seq_len, seq_len)

        print("scores shape:", scores.shape)

        attention_weights = torch.softmax(scores, dim=-1)
        # attention_weights: (batch_size, seq_len, seq_len)

        print("attention_weights shape:", attention_weights.shape)

        output = torch.matmul(attention_weights, V)
        # output: (batch_size, seq_len, d_model)

        print("output shape:", output.shape)

        return output, attention_weights

In [7]:
attn = SelfAttention(d_model=8)
output, weights = attn(x_pos)

print("최종 attention output shape:", output.shape)
print("attention weight shape:", weights.shape)

Q shape: torch.Size([2, 5, 8])
K shape: torch.Size([2, 5, 8])
V shape: torch.Size([2, 5, 8])
scores shape: torch.Size([2, 5, 5])
attention_weights shape: torch.Size([2, 5, 5])
output shape: torch.Size([2, 5, 8])
최종 attention output shape: torch.Size([2, 5, 8])
attention weight shape: torch.Size([2, 5, 5])


## Multi-Head Attention



In [9]:
#assert는 조건이 참인지 검사하고 거짓이면 에러를 내줌.
def process_matrix(mat):
    assert len(mat) == 2   # 행이 2개여야 함.
    assert len(mat[0])==3  #열이 3개여야 합니다.
    assert len(mat[1])==3   #열이 3개여야 합니다.
    print("행렬 크기 확인 완료")

matrix = [
    [1,2,3],
    [4,5,6]
]
process_matrix(matrix)

행렬 크기 확인 완료


In [14]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        # MHA에서 d_model 전체를 여러 had가 같이 나누어 가짐. -> 균등 분할      

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)

        self.fc_out = nn.Linear(d_model, d_model)

    def forward(self, x):
        batch_size, seq_len, d_model = x.shape

        Q = self.W_q(x)  # (B, L, D)
        K = self.W_k(x)
        V = self.W_v(x)

        # (B, L, D) -> (B, L, num_heads, head_dim)
        Q = Q.view(batch_size, seq_len, self.num_heads, self.head_dim)
        K = K.view(batch_size, seq_len, self.num_heads, self.head_dim)
        V = V.view(batch_size, seq_len, self.num_heads, self.head_dim)

        # (B, L, H, Hd) -> (B, H, L, Hd)
        Q = Q.transpose(1, 2)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)

        print("Q shape:", Q.shape)  # (B, H, L, Hd)
        print("K shape:", K.shape)
        print("V shape:", V.shape)

        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)
        # (B, H, L, L)

        attention_weights = torch.softmax(scores, dim=-1)

        out = torch.matmul(attention_weights, V)
        # (B, H, L, Hd)

        # 다시 합치기: (B, H, L, Hd) -> (B, L, H, Hd)
        out = out.transpose(1, 2).contiguous()

        # (B, L, H, Hd) -> (B, L, D)
        out = out.view(batch_size, seq_len, self.d_model)

        out = self.fc_out(out)

        return out, attention_weights

In [11]:
# view: 새로운 텐서를 만드는 것이 아니라, 기존 텐서의 메모리 버터를 공유하는 view(보기)를 생성

import torch
x = torch.arange(12)
y = x.view(3,4)
print(y)
print(x)

tensor([[ 0,  1,  2,  3],
        [ 4,  5,  6,  7],
        [ 8,  9, 10, 11]])
tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])


In [ ]:
# view에 꼭 나오는 개념, contiguous: 메모리상에 요소들이 빈틈없이 나열되어 있다.

import torch

x = torch.randn(3,2)
y = x.transpose(0,1) # 차원을 바꿈(메모리 순서는 그대로)

print(y.is_contiguous()) #Flase
## is_contiguous:  현재 텐서가 메모리에 순서대로 잘 정렬이 되어 있는지? True/False
## x.contiguous(): 만약 텐서가 연속적이지 않으면, 메모리 레이아웃을 새로 정리하여 연속적인 복사본을 만듦

##y.view(-1) -> error 

False


In [15]:
mha = MultiHeadAttention(d_model=8, num_heads=2)
out, attn_weights = mha(x_pos)

print("MHA output shape:", out.shape)
print("MHA attention shape:", attn_weights.shape)

Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
MHA output shape: torch.Size([2, 5, 8])
MHA attention shape: torch.Size([2, 2, 5, 5])


## Feed Forward Network

Attention 다음에는 보통 postion-wise feed forward가 들어감.

In [14]:
class FeedForward(nn.Module):
    def __init__(self, d_model, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model)
        )

    def forward(self, x):
        return self.net(x)

### Add & Norm

x -> sublayer(x) -> x + sublayer(x) -> LayerNorm


## Encoder Block

In [15]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, dropout=0.1):
        super().__init__()

        self.attention = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)

        self.ffn = FeedForward(d_model, hidden_dim)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        attn_out, attn_weights = self.attention(x)
        x = self.norm1(x + self.dropout1(attn_out))

        ffn_out = self.ffn(x)
        x = self.norm2(x + self.dropout2(ffn_out))

        return x, attn_weights

In [16]:
encoder_block = EncoderBlock(d_model=8, num_heads=2, hidden_dim=16)

out, attn_weights = encoder_block(x_pos)

print("EncoderBlock output shape:", out.shape)

Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
EncoderBlock output shape: torch.Size([2, 5, 8])


## 전체 Transformer Encoder

In [17]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, max_len, d_model, num_heads, hidden_dim, num_layers):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = SinusoidalPositionalEncoding(max_len, d_model)

        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, hidden_dim)
            for _ in range(num_layers)
        ])

    def forward(self, x):
        # x: (batch_size, seq_len)
        x = self.embedding(x)
        x = self.pos_encoding(x)

        attention_maps = []

        for layer in self.layers:
            x, attn_weights = layer(x)
            attention_maps.append(attn_weights)

        return x, attention_maps

In [18]:
model = TransformerEncoder(
    vocab_size=20,
    max_len=10,
    d_model=8,
    num_heads=2,
    hidden_dim=16,
    num_layers=2
)

x = torch.tensor([
    [1, 3, 5, 7, 9],
    [2, 4, 6, 8, 0]
])

out, attention_maps = model(x)

print("최종 출력 shape:", out.shape)  # (2, 5, 8)
print("attention map 개수:", len(attention_maps))  # 2개 layer
print("첫 번째 layer attention shape:", attention_maps[0].shape)

Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
최종 출력 shape: torch.Size([2, 5, 8])
attention map 개수: 2
첫 번째 layer attention shape: torch.Size([2, 2, 5, 5])


In [19]:
# 마지막 분류기 설치
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, max_len, d_model, num_heads, hidden_dim, num_layers, num_classes):
        super().__init__()

        self.encoder = TransformerEncoder(
            vocab_size, max_len, d_model, num_heads, hidden_dim, num_layers
        )

        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x, attention_maps = self.encoder(x)
        # x: (B, L, D)

        x = x.mean(dim=1)  # (B, D)
        logits = self.classifier(x)  # (B, num_classes)

        return logits, attention_maps

In [20]:
classifier = TransformerClassifier(
    vocab_size=20,
    max_len=10,
    d_model=8,
    num_heads=2,
    hidden_dim=16,
    num_layers=2,
    num_classes=3
)

x = torch.tensor([
    [1, 3, 5, 7, 9],
    [2, 4, 6, 8, 0]
])

logits, attention_maps = classifier(x)

print("logits shape:", logits.shape)  # (2, 3)
print(logits)

Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
Q shape: torch.Size([2, 2, 5, 4])
K shape: torch.Size([2, 2, 5, 4])
V shape: torch.Size([2, 2, 5, 4])
logits shape: torch.Size([2, 3])
tensor([[ 1.0800, -0.0630, -0.2851],
        [ 0.1259,  0.1908,  0.7315]], grad_fn=<AddmmBackward0>)
